In [ ]:
%pip install interpret==0.7.8

In [ ]:
import mlflow
import interpret
import sklearn
import pandas as pd
import boto3

print("MLflow:", mlflow.__version__)
print("InterpretML:", interpret.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("AWS Region:", boto3.Session().region_name)

In [ ]:
import os
import mlflow

# Store MLflow database in the current notebook directory
DB_PATH = os.path.join(os.getcwd(), "mlflow.db")

# Configure SQLite tracking backend
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Create or retrieve experiment
EXPERIMENT_NAME = "Heart-Disease-Prediction"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow Version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Database Path:", DB_PATH)
print("Experiment Name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

In [ ]:
import os
import boto3
import pandas as pd

BUCKET_NAME = "YOUR_S3_BUCKET_NAME"
S3_KEY = "data/raw/heart_disease_reference.csv"

LOCAL_DIR = "data/raw"
LOCAL_PATH = os.path.join(LOCAL_DIR, "heart_disease_reference.csv")

os.makedirs(LOCAL_DIR, exist_ok=True)

s3 = boto3.client("s3", region_name="ap-southeast-2")

s3.download_file(BUCKET_NAME, S3_KEY, LOCAL_PATH)

df = pd.read_csv(LOCAL_PATH)

print("Dataset downloaded successfully")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Check duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# Check target distribution
print("\nTarget distribution:")
print(df["target"].value_counts())

# Check data types
print("\nData types:")
print(df.dtypes)

In [ ]:
from sklearn.model_selection import train_test_split

# Define input features and target
X = df.drop(columns=["target"])
y = df["target"]

# Feature categories
categorical_features = [
    "sex", "chest-pain", "fasting-blood-sugar",
    "electrocardiographic", "angina", "slope",
    "major-vessels", "thal"
]

continuous_features = [
    "age", "rest-bp", "serum-chol",
    "max-heart-rate", "oldpeak"
]

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

In [ ]:
import mlflow
import mlflow.sklearn

from interpret.glassbox import ExplainableBoostingClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Cross-validation configuration
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define EBM
ebm_model = ExplainableBoostingClassifier(
    interactions=0,
    random_state=42
)

# Define Logistic Regression pipeline
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

print("EBM model configured.")
print("Logistic Regression configured.")
print("5-fold stratified cross-validation configured.")

In [ ]:
# Define feature types exactly as in the original notebook

feature_types = [
    "nominal" if col in categorical_features else "continuous"
    for col in X_train.columns
]

# Reconfigure EBM
ebm_model = ExplainableBoostingClassifier(
    feature_types=feature_types,
    random_state=42,
    interactions=0
)

# Reconfigure Logistic Regression
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print("EBM configuration restored.")
print("Logistic Regression configuration restored.")
print("\nFeature types:", feature_types)

In [ ]:

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

models = {
    "EBM": ebm_model,
    "Logistic_Regression": lr_model
}

cv_results = {}

for model_name, model in models.items():

    print(f"\nEvaluating {model_name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1
    )

    metrics = {
        metric: float(scores[f"test_{metric}"].mean())
        for metric in scoring
    }

    cv_results[model_name] = metrics

    with mlflow.start_run(run_name=f"{model_name}_CV_Corrected"):
        mlflow.log_param("model", model_name)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("random_state", 42)
        mlflow.log_metrics(metrics)

    print(metrics)

cv_results_df = pd.DataFrame(cv_results).T

print("\nCross-validation comparison:")
display(cv_results_df)


In [ ]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

final_models = {
    "EBM": clone(ebm_model),
    "Logistic_Regression": clone(lr_model)
}

test_results = {}
trained_models = {}

for model_name, model in final_models.items():

    print(f"\nTraining {model_name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    test_results[model_name] = metrics
    trained_models[model_name] = model

    cm = confusion_matrix(y_test, y_pred)

    print("Test Metrics:", metrics)
    print("Confusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Log final test results in MLflow
    with mlflow.start_run(run_name=f"{model_name}_Test_Corrected"):

        mlflow.log_param("model", model_name)
        mlflow.log_param("dataset", "UCI_Statlog_Heart_Disease")
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("random_state", 42)

        mlflow.log_metrics({
            f"test_{key}": float(value)
            for key, value in metrics.items()
        })

        mlflow.log_metric("test_true_negative", int(cm[0, 0]))
        mlflow.log_metric("test_false_positive", int(cm[0, 1]))
        mlflow.log_metric("test_false_negative", int(cm[1, 0]))
        mlflow.log_metric("test_true_positive", int(cm[1, 1]))

print("\nFinal Test Comparison:")
display(pd.DataFrame(test_results).T)


In [ ]:

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

display(
    runs[
        ["tags.mlflow.runName", "status",
         "metrics.accuracy",
         "metrics.test_accuracy"]
    ].head(10)
)


In [ ]:

import mlflow
import mlflow.sklearn

all_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

for model_name, model in trained_models.items():

    run_name = f"{model_name}_Test_Corrected"

    matching_runs = all_runs[
        all_runs["tags.mlflow.runName"] == run_name
    ]

    if matching_runs.empty:
        print(f"Run not found: {run_name}")
        continue

    run_id = matching_runs.iloc[0]["run_id"]

    with mlflow.start_run(run_id=run_id):

        if model_name == "EBM":
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
                skops_trusted_types=[
                    "interpret.glassbox._ebm._ebm.ExplainableBoostingClassifier"
                ]
            )
        else:
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model"
            )

    print(f"{model_name} model saved to MLflow.")


In [ ]:
import time
import numpy as np
import pandas as pd

comparable_latency_results = {}

for model_name, model in trained_models.items():

    sample = X_test.iloc[[0]]

    # Warm-up
    for _ in range(20):
        model.predict_proba(sample)

    latencies = []

    # Measure only predict_proba()
    for _ in range(500):
        start = time.perf_counter_ns()

        model.predict_proba(sample)

        end = time.perf_counter_ns()

        latencies.append((end - start) / 1_000_000)

    comparable_latency_results[model_name] = {
        "mean_ms": float(np.mean(latencies)),
        "median_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
        "min_ms": float(np.min(latencies)),
        "max_ms": float(np.max(latencies)),
        "requests": 500,
        "warmup": 20
    }

latency_comparison_df = pd.DataFrame(
    comparable_latency_results
).T

display(latency_comparison_df.round(4))

In [ ]:
import mlflow

for model_name, metrics in comparable_latency_results.items():

    run_name = f"{model_name}_Test_Corrected"

    all_runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    matching_runs = all_runs[
        all_runs["tags.mlflow.runName"] == run_name
    ]

    if matching_runs.empty:
        print(f"Run not found: {run_name}")
        continue

    run_id = matching_runs.iloc[0]["run_id"]

    with mlflow.start_run(run_id=run_id):
        mlflow.log_metrics({
            f"comparable_latency_{key}": float(value)
            for key, value in metrics.items()
            if key not in ["requests", "warmup"]
        })

        mlflow.set_tag("latency_benchmark_method", "predict_proba_only")
        mlflow.set_tag("latency_warmup_iterations", "20")
        mlflow.set_tag("latency_measurement_iterations", "500")

    print(f"{model_name}: comparable latency metrics logged.")

In [ ]:
import mlflow

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

verification = runs[
    runs["tags.mlflow.runName"].isin([
        "EBM_Test_Corrected",
        "Logistic_Regression_Test_Corrected"
    ])
]

columns = [
    "tags.mlflow.runName",
    "status",
    "metrics.test_accuracy",
    "metrics.comparable_latency_mean_ms",
    "metrics.comparable_latency_median_ms",
    "metrics.comparable_latency_p95_ms",
    "metrics.comparable_latency_p99_ms"
]

display(verification[columns])

In [ ]:
import boto3

s3 = boto3.client("s3")

BUCKET_NAME = "YOUR_S3_BUCKET_NAME"

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="artifacts/",
    MaxKeys=30
)

for obj in response.get("Contents", []):
    print(obj["Key"], round(obj["Size"] / 1024, 2), "KB")

In [ ]:
import os
import mlflow

EXPORT_DIR = "artifacts/mlflow_corrected"
os.makedirs(EXPORT_DIR, exist_ok=True)

for model_name in ["EBM", "Logistic_Regression"]:

    run_name = f"{model_name}_Test_Corrected"

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    matching = runs[
        runs["tags.mlflow.runName"] == run_name
    ]

    if matching.empty:
        print(f"Run not found: {run_name}")
        continue

    run_id = matching.iloc[0]["run_id"]

    model_uri = f"runs:/{run_id}/model"

    local_path = mlflow.artifacts.download_artifacts(
        artifact_uri=model_uri,
        dst_path=EXPORT_DIR
    )

    print(f"{model_name} exported to: {local_path}")

In [ ]:
import os
import shutil

EXPORT_DIR = "artifacts/mlflow_corrected"

ebm_source = os.path.join(EXPORT_DIR, "model")
ebm_destination = os.path.join(EXPORT_DIR, "ebm_model")

if os.path.exists(ebm_destination):
    shutil.rmtree(ebm_destination)

shutil.move(ebm_source, ebm_destination)

# Download Logistic Regression again into a separate folder
lr_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

lr_run_id = lr_runs.loc[
    lr_runs["tags.mlflow.runName"] == "Logistic_Regression_Test_Corrected",
    "run_id"
].iloc[0]

lr_destination = os.path.join(EXPORT_DIR, "logistic_model")

mlflow.artifacts.download_artifacts(
    artifact_uri=f"runs:/{lr_run_id}/model",
    dst_path=lr_destination
)

print("Exported model folders:")
print(os.listdir(EXPORT_DIR))

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime, timezone

EXPORT_DIR = "artifacts/mlflow_corrected"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Feature information
with open(os.path.join(EXPORT_DIR, "feature_columns.json"), "w") as f:
    json.dump(list(X_train.columns), f, indent=4)

with open(os.path.join(EXPORT_DIR, "categorical_features.json"), "w") as f:
    json.dump(categorical_features, f, indent=4)

# Test metrics
test_metrics = {
    "EBM": {
        "accuracy": 0.8703703704,
        "precision": 0.8148148148,
        "recall": 0.9166666667,
        "f1": 0.8627450980,
        "roc_auc": 0.9083333333
    },
    "Logistic_Regression": {
        "accuracy": 0.8518518519,
        "precision": 0.7857142857,
        "recall": 0.9166666667,
        "f1": 0.8461538462,
        "roc_auc": 0.8986111111
    }
}

with open(os.path.join(EXPORT_DIR, "test_metrics.json"), "w") as f:
    json.dump(test_metrics, f, indent=4)

# Comparable latency metrics
with open(os.path.join(EXPORT_DIR, "latency_metrics.json"), "w") as f:
    json.dump(comparable_latency_results, f, indent=4)

# Project metadata
metadata = {
    "project": "Explainable and Drift-Aware MLOps Pipeline for Heart Disease Prediction",
    "dataset": "UCI Statlog Heart Disease",
    "primary_model": "Explainable Boosting Machine",
    "baseline_model": "Logistic Regression",
    "training_environment": "Amazon SageMaker Studio",
    "tracking_tool": "MLflow",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "latency_method": "predict_proba only; 20 warmups; 500 measurements",
    "note": "Educational research prototype; not for clinical diagnosis."
}

with open(os.path.join(EXPORT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Prepared files:")
print(os.listdir(EXPORT_DIR))

In [ ]:
import os

BUCKET_NAME = "YOUR_S3_BUCKET_NAME"
LOCAL_DIR = "artifacts/mlflow_corrected"
S3_PREFIX = "artifacts/mlflow_corrected/"

for root, dirs, files in os.walk(LOCAL_DIR):
    for filename in files:
        local_path = os.path.join(root, filename)

        relative_path = os.path.relpath(local_path, LOCAL_DIR)
        s3_key = S3_PREFIX + relative_path.replace("\\", "/")

        s3.upload_file(
            local_path,
            BUCKET_NAME,
            s3_key
        )

        print(f"Uploaded: {s3_key}")

print("\nUpload completed.")

In [ ]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="artifacts/mlflow_corrected/"
)

print("Files stored in S3:\n")

for obj in response.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size'] / 1024:.2f} KB)")

print("\nTotal files:", len(response.get("Contents", [])))

In [ ]:
%pip install evidently

In [ ]:
import evidently
import numpy as np
import sklearn

print("Evidently:", evidently.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

In [ ]:
import os

os.makedirs("data/monitoring", exist_ok=True)

reference_data = X_train.copy()

reference_data.to_csv(
    "data/monitoring/reference_data.csv",
    index=False
)

print("Reference dataset saved.")
print("Shape:", reference_data.shape)
print("\nMissing values:", reference_data.isnull().sum().sum())
print("\nFirst five rows:")
display(reference_data.head())

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Normal production data
normal_data = X_train.sample(
    n=150,
    replace=True,
    random_state=42
).reset_index(drop=True)

# Drifted production data
drifted_data = normal_data.copy()

# Introduce controlled distribution changes
drifted_data["age"] += 10
drifted_data["serum-chol"] += 40
drifted_data["max-heart-rate"] -= 20
drifted_data["oldpeak"] += 1.0

# Save both datasets
normal_data.to_csv(
    "data/monitoring/normal_production_data.csv",
    index=False
)

drifted_data.to_csv(
    "data/monitoring/drifted_production_data.csv",
    index=False
)

print("Normal production shape:", normal_data.shape)
print("Drifted production shape:", drifted_data.shape)

print("\nReference means:")
display(reference_data.mean().round(2))

print("\nDrifted production means:")
display(drifted_data.mean().round(2))

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

# Create drift report for normal production data
normal_report = Report([
    DataDriftPreset()
])

normal_result = normal_report.run(
    reference_data=reference_data,
    current_data=normal_data
)

normal_result.save_html(
    "data/monitoring/normal_drift_report.html"
)

# Create drift report for drifted production data
drifted_report = Report([
    DataDriftPreset()
])

drifted_result = drifted_report.run(
    reference_data=reference_data,
    current_data=drifted_data
)

drifted_result.save_html(
    "data/monitoring/drifted_report.html"
)

print("Both drift reports generated successfully.")

In [ ]:
DRIFT_THRESHOLD = 0.20

normal_drift_share = normal_dict["metrics"][0]["value"]["share"]
drifted_share = drifted_dict["metrics"][0]["value"]["share"]

def drift_decision(drift_share):
    if drift_share >= DRIFT_THRESHOLD:
        return "DRIFT_ALERT"
    return "NORMAL"

normal_status = drift_decision(normal_drift_share)
drifted_status = drift_decision(drifted_share)

print("Normal production status:", normal_status)
print("Normal drift share:", round(normal_drift_share, 4))

print("\nDrifted production status:", drifted_status)
print("Drifted share:", round(drifted_share, 4))

In [ ]:
import os
import json
from datetime import datetime, timezone

monitoring_results = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "reference_dataset": "UCI Statlog Heart Disease training data",
    "normal_production": {
        "drift_share": normal_drift_share,
        "status": normal_status
    },
    "drifted_production": {
        "drift_share": drifted_share,
        "status": drifted_status
    },
    "alert_threshold": DRIFT_THRESHOLD,
    "monitoring_tool": "Evidently",
    "evidently_version": evidently.__version__
}

with open(
    "data/monitoring/drift_monitoring_results.json",
    "w"
) as f:
    json.dump(monitoring_results, f, indent=4)

print(json.dumps(monitoring_results, indent=4))

In [ ]:
import os

MONITORING_DIR = "data/monitoring"
S3_PREFIX = "artifacts/monitoring/"

for root, dirs, files in os.walk(MONITORING_DIR):
    for filename in files:
        local_path = os.path.join(root, filename)

        relative_path = os.path.relpath(
            local_path, MONITORING_DIR
        )

        s3_key = S3_PREFIX + relative_path.replace("\\", "/")

        s3.upload_file(
            local_path,
            BUCKET_NAME,
            s3_key
        )

        print("Uploaded:", s3_key)

print("\nMonitoring artifacts uploaded.")

In [ ]:
import boto3

sts = boto3.client("sts")
print("AWS identity:")
print(sts.get_caller_identity())

for service in ["lambda", "apigateway"]:
    try:
        if service == "lambda":
            client = boto3.client("lambda")
            client.list_functions(MaxItems=1)
        else:
            client = boto3.client("apigateway")
            client.get_rest_apis(limit=1)

        print(f"{service}: Read access available")

    except Exception as e:
        print(f"{service}: {type(e).__name__}: {e}")

In [ ]:
import boto3

iam = boto3.client("iam")

role_arn = (
    "YOUR_SAGEMAKER_EXECUTION_ROLE_ARN"
    ""
)

actions = [
    "lambda:CreateFunction",
    "lambda:UpdateFunctionCode",
    "lambda:AddPermission",
    "apigateway:POST",
    "iam:PassRole",
    "logs:CreateLogGroup",
    "logs:PutLogEvents"
]

try:
    result = iam.simulate_principal_policy(
        PolicySourceArn=role_arn,
        ActionNames=actions
    )

    for item in result["EvaluationResults"]:
        print(
            item["EvalActionName"],
            ":", item["EvalDecision"]
        )

except Exception as e:
    print(type(e).__name__, e)

In [ ]:
import boto3

ec2 = boto3.client("ec2", region_name="ap-southeast-2")

try:
    response = ec2.describe_instances(MaxResults=5)
    print("EC2 read access available")

except Exception as e:
    print(type(e).__name__, e)

In [ ]:
import boto3

ec2 = boto3.client("ec2", region_name="ap-southeast-2")

INSTANCE_ID = "i-06bfaba3b5af8078e"

response = ec2.describe_instances(
    InstanceIds=[INSTANCE_ID]
)

for reservation in response["Reservations"]:
    for instance in reservation["Instances"]:
        print("Instance ID:", instance["InstanceId"])
        print("State:", instance["State"]["Name"])
        print("Instance Type:", instance["InstanceType"])
        print("Public IP:", instance.get("PublicIpAddress"))
        print("Private IP:", instance.get("PrivateIpAddress"))
        print("Security Groups:", instance["SecurityGroups"])

In [ ]:
import boto3

iam = boto3.client("iam")

role_name = "HeartDiseaseEC2S3Role"

print("Attached policies:")
policies = iam.list_attached_role_policies(
    RoleName=role_name
)

for policy in policies["AttachedPolicies"]:
    print(policy["PolicyName"], policy["PolicyArn"])

print("\nInline policies:")
inline = iam.list_role_policies(RoleName=role_name)

for policy_name in inline["PolicyNames"]:
    print(policy_name)

In [ ]:
import boto3
import json

iam = boto3.client("iam")

policy = iam.get_role_policy(
    RoleName="HeartDiseaseEC2S3Role",
    PolicyName="HeartDiseaseCloudWatchLogsPolicy"
)

print(json.dumps(policy["PolicyDocument"], indent=2))

In [ ]:
import mlflow

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print(
    runs[
        runs["tags.mlflow.runName"].isin([
            "EBM_Test_Corrected",
            "Logistic_Regression_Test_Corrected"
        ])
    ][["tags.mlflow.runName", "run_id", "status"]]
)

In [ ]:
import mlflow

EBM_RUN_ID = "a7fea3db74ad47c8b9d0fa85d7997747"

ebm_model_uri = f"runs:/{EBM_RUN_ID}/model"

ebm_local_path = mlflow.artifacts.download_artifacts(
    artifact_uri=ebm_model_uri,
    dst_path="artifacts/verified_ebm"
)

print("Downloaded EBM artifact:", ebm_local_path)

In [ ]:
import mlflow.sklearn

verified_ebm = mlflow.sklearn.load_model(ebm_local_path)

print("Model class:", type(verified_ebm))

In [ ]:
import boto3
import os

bucket = "YOUR_S3_BUCKET_NAME"

local_dir = ebm_local_path
s3_prefix = "artifacts/verified_ebm/model/"

s3 = boto3.client("s3", region_name="ap-southeast-2")

for root, dirs, files in os.walk(local_dir):
    for filename in files:
        local_path = os.path.join(root, filename)
        relative_path = os.path.relpath(local_path, local_dir)
        s3_key = s3_prefix + relative_path.replace("\\", "/")

        s3.upload_file(local_path, bucket, s3_key)
        print("Uploaded:", s3_key)

print("Verified EBM model uploaded successfully!")

In [ ]:
import os

db_path = "/home/sagemaker-user/mlflow.db"

print("Database exists:", os.path.exists(db_path))
if os.path.exists(db_path):
    print("Database size:", os.path.getsize(db_path), "bytes")

In [ ]:
import mlflow
print("MLflow version:", mlflow.__version__)